# Exploración SPlink con datos de Comorbilidad de COVID-19

Esta es una exploración de la biblioteca Splink siguiendo el [tutorial](https://moj-analytical-services.github.io/splink/demos/tutorials/00_Tutorial_Introduction.html) que provee la documentación y usando la base de datos:

INER_COVID19_Pacientes_DiagnosticoComorbilidad.csv


In [1]:
# Llamado de dependencias y carga de datos
import os
import pandas as pd

ruta_datos = os.path.join("data", "INER_COVID19_Pacientes_DiagnosticoComorbilidad.csv")
df = pd.read_csv(ruta_datos, encoding="utf-8")

In [2]:
df.head(5)

,expediente,nombre,fechaing,fechaegr,diagnosticoprincipal,cie101,diagnostico2,cie102,diagnostico3,cie103,...,dx4,obesidad,obesidad1,cardiopatia,comorbi,diabetes,nefropatia,eaperge,tephap,comorbicv
0,237124,ANGEL OCTAVIO GUERRERO SANCHEZ,2020-02-28,2020-03-02,COVID 19,U07.1,NODULO PULMONAR,J98.4,NaN,NaN,...,NaN,0.0,0.0,0.0,Otras,0.0,0.0,0.0,0.0,0.0
1,237346,FELIPE GONZALEZ GUTIERREZ,2020-03-14,2020-03-18,SARS COV2,U07.1,NaN,NaN,NaN,NaN,...,NaN,0.0,0.0,0.0,Ninguna,0.0,0.0,0.0,0.0,0.0
2,237349,MARIANA ARCEO GUTIERREZ,2020-03-16,2020-03-21,COVID 19,U07.1,NEUMONIA ADQUIRIDA EN LA COMUNIDAD,J18.9,INSUFICIENCIA RESPIRATORIA TIPO I,J96.0,...,NaN,0.0,0.0,0.0,Otras,0.0,0.0,0.0,0.0,0.0
3,237380,MARTHA DURAN IBARRA,2020-03-18,2020-03-24,SOSPECHA DE COVID 19,U07.2,NEUMONIA POR METAPNEUMOVIRUS,J12.8,DIABETES MELLITUS TIPO II,E11.9,...,NaN,0.0,0.0,0.0,Diabetes Mellitus,1.0,0.0,0.0,0.0,0.0
4,237399,ISRAEL KRAUSS CELAYA,2020-03-19,2020-03-24,SOSPECHA DE COVID 19,U07.2,NEUMONIA ADQUIRIDA EN LA COMUNIDAD,J18.9,INSUFICIENCIA RESPIRATORIA TIPO I,J96.0,...,NaN,0.0,0.0,0.0,Otras,0.0,0.0,0.0,0.0,0.0


In [3]:
len(df)

4278

## Preliminares

Splink requiere bases de datos lo más limpias posibles. En este caso se añade un identificador único consecutivo (`unique_id`) a la tabla, además de que se limpian espacios extras en blanco al inicio, fin y entre cada nombre completo de la columna `nombre`.

In [4]:
# Añadiendo un ID
df["unique_id"] = range(1, len(df) + 1)
df.insert(0, "unique_id", df.pop("unique_id"))

In [5]:
# Limpiando espacios en la columna nombre
df["nombre"] = df["nombre"].str.replace(r"\s+", " ", regex=True).str.strip()

## 2. Análisis Exploratorio de Datos usando Splink




Splink provee de herramientas para el análisis exploratorio de datos que facilitarán la elección de reglas de agrupamiento en los siguientes pasos. Exploro aquí las que se usan en el tutorial de la biblioteca.

### Datos faltantes


In [6]:
from splink.exploratory import completeness_chart
from splink import DuckDBAPI
db_api = DuckDBAPI()
completeness_chart(df, db_api=db_api)

alt.LayerChart(...)

### Distribución de los valores en los datos


In [7]:
from splink.exploratory import profile_columns

profile_columns(df, db_api=DuckDBAPI(), top_n=10, bottom_n=5)

alt.VConcatChart(...)

## 3. Escoger las reglas de bloqueo

Se usan las reglas de bloqueo para generar pares de registros candidatos a comparar. Según la [documentación el objetivo de éstas es doble](https://moj-analytical-services.github.io/splink/demos/tutorials/03_Blocking.html#devising-effective-blocking-rules-for-prediction):

1. Eliminar suficientes pares de comparación que no coincidan para que el proceso de vinculación de registros sea lo suficientemente pequeño para que pueda calcularse.

2. Eliminar la menor cantidad posible de pares que coincidan realmente (idealmente ninguno).

Splink recomienda la generación de múltiples reglas de bloqueo para lograr ambos objetivos, por lo cual podrían ser entre 3 y 10 reglas de bloqueo.

En el análisis exploratorio del segundo paso, se observa que hay un nombre de un paciente que es el registro más repetido tanto en la columna `expediente`como en la columna `nombre` y que entonces estos cinco registros podrían ser una sola persona.

Entonces los candidatos a las reglas de bloqueo para deduplicar estos cinco registros y demás que se repitan pueden ser:

1. `expediente` y `nombre`
2. `expediente` y `obesidad`
3. `expediente` y `cardiopatia`
4. `expediente` y `diabetes`

O tal vez cualquier combinación de `expediente` y alguna otra columna que indique enfermedad.

In [8]:
# Candidatos a reglas de bloqueo
from splink import block_on
block_on("expediente", "nombre")
block_on("expediente", "obesidad")
block_on("expediente", "cardiopatia")
block_on("expediente", "diatebes")

Para saber si los candidatos a las reglas de bloqueo funcionarán, Splink tiene diveras herramientas para ayudar a elegir.

### Contar el número de comparaciones creadas por una sóla regla de bloqueo

El número de comparaciones que genera una regla de bloqueo crece de forma cuadrática respecto al número de registros. El número de pares en una [link text](https://)tabla con N registros está dado por N(N-1)/2

In [9]:
from splink.blocking_analysis import count_comparisons_from_blocking_rule

db_api = DuckDBAPI()

br_nombre = block_on("expediente", "nombre")

counts = count_comparisons_from_blocking_rule(
    table_or_tables=df,
    blocking_rule=br_nombre,
    link_type="dedupe_only",
    db_api=db_api,
)

counts

{'number_of_comparisons_generated_pre_filter_conditions': 4660,
 'number_of_comparisons_to_be_scored_post_filter_conditions': 191,
 'filter_conditions_identified': '',
 'equi_join_conditions_identified': 'l."expediente" = r."expediente" AND l."nombre" = r."nombre"',
 'link_type_join_condition': 'where l."unique_id" < r."unique_id"'}

In [10]:
br_obesidad = block_on("expediente", "obesidad")

counts = count_comparisons_from_blocking_rule(
    table_or_tables=df,
    blocking_rule=br_obesidad,
    link_type="dedupe_only",
    db_api=db_api,
)

counts

{'number_of_comparisons_generated_pre_filter_conditions': 4644,
 'number_of_comparisons_to_be_scored_post_filter_conditions': 183,
 'filter_conditions_identified': '',
 'equi_join_conditions_identified': 'l."expediente" = r."expediente" AND l."obesidad" = r."obesidad"',
 'link_type_join_condition': 'where l."unique_id" < r."unique_id"'}

### _Worst offending values_

In [11]:
from splink.blocking_analysis import n_largest_blocks

result = n_largest_blocks(    table_or_tables=df,
    blocking_rule= block_on("expediente", "nombre"),
    link_type="dedupe_only",
    db_api=db_api,
    n_largest=10
    )

result.as_pandas_dataframe()

,key_0,key_1,count_l,count_r,block_count
0,249200,VICENTE MARTIN VALENCIA CHAVEZ,5,5,25
1,242473,ALBERTO GARCIA RAMIREZ,4,4,16
2,238376,JAVIER RODRIGUEZ ROJAS,4,4,16
3,237871,JOSE EMILIANO GONZALEZ LARA,3,3,9
4,240503,GUADALUPE NAVA RODRIGUEZ,3,3,9
5,239088,RUBEN SECUNDINO AGAPITO,3,3,9
6,239109,AIDA IMELDA VALERO CHAVEZ,3,3,9
7,114639,GABINO LEMUS HERNANDEZ,3,3,9
8,241590,JORGE ARTURO VELAZQUEZ CARRANZA,3,3,9
9,239197,FELIPA ROSALINA MONTES MEZA,3,3,9


In [12]:
from splink.blocking_analysis import n_largest_blocks

result = n_largest_blocks(    table_or_tables=df,
    blocking_rule= block_on("expediente", "diabetes"),
    link_type="dedupe_only",
    db_api=db_api,
    n_largest=10
    )

result.as_pandas_dataframe()

,key_0,key_1,count_l,count_r,block_count
0,249200,0.0,5,5,25
1,238904,0.0,4,4,16
2,242473,0.0,4,4,16
3,241590,0.0,3,3,9
4,237871,0.0,3,3,9
5,114639,0.0,3,3,9
6,239197,0.0,3,3,9
7,238743,0.0,3,3,9
8,191202,0.0,3,3,9
9,238389,0.0,3,3,9


### Contar el número de comparaciones creadas por una lista de reglas de agrupamiento

Genera una gráfica de comparaciones acumuladas donde cada barra representa cuántas comparaciones nuevas y únicas aporta cada regla adicional, después de desduplicar contra las reglas anteriores.

- Si una regla aporta muy pocas comparaciones nuevas y únicas significa que casi todos los pares de registros nuevos que genera ya estaban cubiertos por alguna regla o reglas anteriores, es decir, es redundante.
- Si una regla aporta muchas más comparaciones nuevas y únicas significa que está capturando pares que las otras reglas no cubren, por lo tanto, es valiiosa.
- El acumulado final me indica si la lista de reglas es manejable computacionalmente.

In [13]:
from splink.blocking_analysis import (
    cumulative_comparisons_to_be_scored_from_blocking_rules_chart,
)

blocking_rules_for_analysis = [
    block_on("expediente", "nombre"),
    block_on("expediente", "obesidad"),
    block_on("expediente", "diabetes"),
    block_on("expediente", "cardiopatia")
]


cumulative_comparisons_to_be_scored_from_blocking_rules_chart(
    table_or_tables=df,
    blocking_rules=blocking_rules_for_analysis,
    db_api=db_api,
    link_type="dedupe_only",
)

alt.Chart(...)

De esta gráfica podemos concluir que mis reglas de bloqueo pueden ser un poco redundantes, ya que el tamaño de la segunda y tercer barras es pequeño, es decir no se generan nuevos pares a comparar. Lo ideal es elegir reglas de agrupamiento que generen más pares a comparar. Pero tal vez esto no es tan grave, mucho depende de la naturaleza del conjunto de datos. En este caso dado que un paciente con diagnóstico de covid puede presentar las mismas comorbilidades, es decir las 4 columnas están altamente correlacionadas, no se esperaba que hubiera más pares nuevos a comparar.

## 4. Estimar los parámetros del modelo

Dada las conclusiones de las herramientas anteriores, en este punto voy a modificar un poco mis reglas de bloqueo y agregar fecha de ingreso y egreso:

In [14]:
# Candidatos a reglas de bloqueo
from splink import block_on
block_on("expediente", "nombre")
block_on("expediente", "obesidad")
block_on("expediente", "cardiopatia")
block_on("expediente", "diabetes")
block_on("nombre", "fechaing")
block_on("nombre", "fechaegr")

In [15]:
# Con este nuevo bloque
blocking_rules_for_analysis = [
block_on("expediente", "nombre"),
block_on("expediente", "obesidad"),
block_on("expediente", "cardiopatia"),
block_on("expediente", "diabetes"),
block_on("nombre", "fechaing"),
block_on("nombre", "fechaegr")
]


cumulative_comparisons_to_be_scored_from_blocking_rules_chart(
    table_or_tables=df,
    blocking_rules=blocking_rules_for_analysis,
    db_api=db_api,
    link_type="dedupe_only",
)

alt.Chart(...)

### Comparaciones (`Comparison`)

Una Comparación o *Comparison* representa como se va a evaluar la similitud de un campo. El modelo que se construye usando Splink consta de muchas comparaciones.

Las comparaciones tienen niveles `ComparisonLevels` donde se asignan calificaciones de similitud entre las columnas para cierta comparación. Por ejemplo:




```
Modelo de vinculación de datos
├─-- Comparison: fechaing
│    ├─-- ComparisonLevel: Coincidencia exacta
│    ├─-- ComparisonLevel: Un caracter de diferencia
│    ├─-- ComparisonLevel: Cualquier otra
├─-- Comparison: nombre
│    ├─-- ComparisonLevel: Coincidencia exacta
│    ├─-- ComparisonLevel: JaroWinkler > 0.9
│    ├─-- ComparisonLevel: Cualquier otra
│    etc.
```



Para `fechaing` o pienso que en particular para cualquier fecha
| fechaing_l      | fechaegr_r      | comparison_level        | interpretation |
|------------|------------|--------------------------|-----------------|
| 1971-05-24 | 1971-05-24 | Coincidencia exacta            | great match     |
| 1971-05-24 | 1971-06-24 | Un caracter de diferencia | fuzzy match     |
| 1971-05-24 | 2000-01-02 | Cualquier otra                | bad match       |

Para `nombre`

| nombre_l | nombre_r | comparison_level | interpretation                                    |
|-----------|-----------|-------------------|----------------------------------------------------|
| VICENTE MARTIN VALENCIA CHAVEZ       | VICENTE MARTIN VALENCIA CHAVEZ       | Exact match       | great match |
| MARTIN VALENCIA CHAVEZ       | VICENTE MARTIN VALENCIA CHAVEZ    | All JaroWinkler >0.9         | great match?  
| VICENTE MARTIN VLENCIA CHAVEZ       | JOSE PEDRO VALENCIA CHAVEZ    | other         | bad match, this comparison has no notion of nicknames

En el segundo caso de nombre donde difiere por el segundo nombre el registro, no es tan conveniente usar `JaroWinkler`, ya que este método le da peso al inicio del string y al faltar el primer nombre de Vicente, la comparación ya no es tan ideal. Para ello Splink tiene un módulo de comparaciones "out of the box" que incluyen estos casos, así como también se pueden customizar las reglas para hacer comparaciones.

### Especificar el modelo usando las comparaciones

Categoría 1: Funciones genéricas que aplican una función de fuzzy matching en particular. Por ejemplo, distancia de Levenshtein.

In [16]:
import splink.comparison_library as cl

nombre_comparacion = cl.LevenshteinAtThresholds("nombre", 2)
print(nombre_comparacion.get_comparison("duckdb").human_readable_description)

Comparison 'LevenshteinAtThresholds' of "nombre".
Similarity is assessed using the following ComparisonLevels:
    - 'nombre is NULL' with SQL rule: "nombre_l" IS NULL OR "nombre_r" IS NULL
    - 'Exact match on nombre' with SQL rule: "nombre_l" = "nombre_r"
    - 'Levenshtein distance of nombre <= 2' with SQL rule: levenshtein("nombre_l", "nombre_r") <= 2
    - 'All other comparisons' with SQL rule: ELSE



Categoría 2: Funciones de comparación diseñadas para tipos de datos específicos.

In [17]:
nombre_completo_comparacion = cl.NameComparison("nombre")
print(nombre_completo_comparacion.get_comparison("duckdb").human_readable_description)

Comparison 'NameComparison' of "nombre".
Similarity is assessed using the following ComparisonLevels:
    - 'nombre is NULL' with SQL rule: "nombre_l" IS NULL OR "nombre_r" IS NULL
    - 'Exact match on nombre' with SQL rule: "nombre_l" = "nombre_r"
    - 'Jaro-Winkler distance of nombre >= 0.92' with SQL rule: jaro_winkler_similarity("nombre_l", "nombre_r") >= 0.92
    - 'Jaro-Winkler distance of nombre >= 0.88' with SQL rule: jaro_winkler_similarity("nombre_l", "nombre_r") >= 0.88
    - 'Jaro-Winkler distance of nombre >= 0.7' with SQL rule: jaro_winkler_similarity("nombre_l", "nombre_r") >= 0.7
    - 'All other comparisons' with SQL rule: ELSE



### Especificar el diccionario de configuraciones

In [18]:
# Antes para poder comparar las fechas de ingreso y egreso voy a customizar un poco la función DateOfBirthComparison

fecha_ingreso_comparacion = cl.DateOfBirthComparison(
    "fechaing",
    input_is_string=True,
    datetime_format="%Y-%m-%d",
    invalid_dates_as_null=True,
    datetime_thresholds=[1, 7, 365],       # límites en días
    datetime_metrics=["day", "day", "day"] # unidad de cada threshold
)

fecha_egreso_comparacion = cl.DateOfBirthComparison(
    "fechaegr",
    input_is_string=True,
    datetime_format="%Y-%m-%d",
    invalid_dates_as_null=True,
    datetime_thresholds=[1, 7, 365],       # límites en días
    datetime_metrics=["day", "day", "day"] # unidad de cada threshold
)

In [19]:
from splink import Linker, SettingsCreator, block_on, DuckDBAPI

settings = SettingsCreator(
    link_type="dedupe_only",
    comparisons=[
        cl.NameComparison("nombre"),
        fecha_ingreso_comparacion,
        fecha_egreso_comparacion
    ],
    blocking_rules_to_generate_predictions=[
        block_on("nombre", "fechaing"),
        block_on("nombre", "fechaegr"),
    ],
    retain_intermediate_calculation_columns=True,
)

linker = Linker(df, settings, db_api=DuckDBAPI())

### Estimar los parámetros del modelo

a. El parámetro `probability_two_random_records_match` es la probabilidad de que dos registros tomados al azar de tus datos de entrada representen un match (típicamente un número muy pequeño).

b. Los valores `u` son la proporción de registros que caen en cada `ComparisonLevel` entre los registros que verdaderamente no son coincidencia.

c. Los valores `m` son la proporción de registros que caen en cada `ComparisonLevel` entre los registros que verdaderamente sí son coincidencia.

In [20]:
# a. probability_two_random_records_matchdf_predictions

deterministic_rules = [
    block_on("nombre", "expediente")
]

linker.training.estimate_probability_two_random_records_match(deterministic_rules, recall=0.7)

Probability two random records match is estimated to be  2.98e-05.
This means that amongst all possible pairwise record comparisons, one in 33,528.55 are expected to match.  With 9,148,503 total possible comparisons, we expect a total of around 272.86 matching pairs


In [21]:
# b.
linker.training.estimate_u_using_random_sampling(max_pairs=1e6)

You are using the default value for `max_pairs`, which may be too small and thus lead to inaccurate estimates for your model's u-parameters. Consider increasing to 1e8 or 1e9, which will result in more accurate estimates, but with a longer run time.
----- Estimating u probabilities using random sampling -----

Estimated u probabilities using random sampling

Your model is not yet fully trained. Missing estimates for:
    - nombre (no m values are trained).
    - fechaing (no m values are trained).
    - fechaegr (no m values are trained).


In [22]:
# c.
linker.training.estimate_m_from_label_column("expediente")

------ Estimating m probabilities using from column expediente -------
m probability not trained for fechaing - Abs date difference <= 1 day (comparison vector value: 3). This usually means the comparison level was never observed in the training data.
m probability not trained for fechaegr - Exact match on date of birth (comparison vector value: 5). This usually means the comparison level was never observed in the training data.
m probability not trained for fechaegr - Abs date difference <= 1 day (comparison vector value: 3). This usually means the comparison level was never observed in the training data.

Your model is not yet fully trained. Missing estimates for:
    - fechaing (some m values are not trained).
    - fechaegr (some m values are not trained).


In [23]:
#c.
sesion_2 = block_on("nombre")
linker.training.estimate_parameters_using_expectation_maximisation(sesion_2)


----- Starting EM training session -----

Estimating the m probabilities of the model by blocking on:
l."nombre" = r."nombre"

Parameter estimates will be made for the following comparison(s):
    - fechaing
    - fechaegr

Parameter estimates cannot be made for the following comparison(s) since they are used in the blocking rules: 
    - nombre

Level Abs date difference <= 1 day on comparison fechaing not observed in dataset, unable to train m value

Level Exact match on date of birth on comparison fechaegr not observed in dataset, unable to train m value

Level Abs date difference <= 1 day on comparison fechaegr not observed in dataset, unable to train m value

Iteration 1: Largest change in params was -0.95 in the m_probability of fechaegr, level `Exact match on date of birth`
Iteration 2: Largest change in params was 0.0684 in probability_two_random_records_match
Iteration 3: Largest change in params was 0.0403 in probability_two_random_records_match
Iteration 4: Largest change i

<EMTrainingSession, blocking on l."nombre" = r."nombre", deactivating comparisons nombre>

### Visualizando los pesos

In [24]:
linker.visualisations.match_weights_chart()

/home/pradel/Desktop/05_utumno/00_lentigob/lentigob-splink/exploracion/.venv/lib/python3.10/site-packages/altair/vegalite/v6/api.py:4138: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  return _tp.from_dict(dct, validate=validate)


alt.VConcatChart(...)

In [25]:
linker.visualisations.m_u_parameters_chart()


alt.HConcatChart(...)

## 5. Predicción

In [26]:
df_predictions = linker.inference.predict(threshold_match_probability=0.2)
df_predictions.as_pandas_dataframe(limit=25)

Blocking time: 0.01 seconds
Predict time: 0.10 seconds

 -- WARNING --
You have called predict(), but there are some parameter estimates which have neither been estimated or specified in your settings dictionary.  To produce predictions the following untrained trained parameters will use default values.
Comparison: 'fechaing':
    m values not fully trained
Comparison: 'fechaegr':
    m values not fully trained


,match_weight,match_probability,unique_id_l,unique_id_r,nombre_l,nombre_r,gamma_nombre,tf_nombre_l,tf_nombre_r,bf_nombre,bf_tf_adj_nombre,fechaing_l,fechaing_r,gamma_fechaing,bf_fechaing,fechaegr_l,fechaegr_r,gamma_fechaegr,bf_fechaegr,match_key
0,1.239486,0.702482,4114,4116,VICENTE MARTIN VALENCIA CHAVEZ,VICENTE MARTIN VALENCIA CHAVEZ,4,0.001169,0.001169,55854.991328,0.013999,2023-03-04,2023-03-04,5,24.081991,2023-03-04,2023-03-06,4,4.204109,0
1,0.917727,0.653876,1544,1888,CARLOS PONCE CHAVEZ,CARLOS PONCE CHAVEZ,4,0.000468,0.000468,55854.991328,0.034997,2020-12-30,2020-12-30,5,24.081991,2021-01-05,2021-02-23,1,1.345473,0
2,0.917727,0.653876,2069,2181,JUAN VICTOR NU?EZ JUAREZ,JUAN VICTOR NU?EZ JUAREZ,4,0.000468,0.000468,55854.991328,0.034997,2021-03-14,2021-03-14,5,24.081991,2021-03-17,2021-04-01,1,1.345473,0
3,0.917727,0.653876,1938,2202,ALFREDO SARABIA CABALLERO,ALFREDO SARABIA CABALLERO,4,0.000468,0.000468,55854.991328,0.034997,2021-03-01,2021-03-01,5,24.081991,2021-03-01,2021-04-03,1,1.345473,0
4,0.917727,0.653876,2324,2570,JOSE HUGO PEREZ SALINAS,JOSE HUGO PEREZ SALINAS,4,0.000468,0.000468,55854.991328,0.034997,2021-04-12,2021-04-12,5,24.081991,2021-04-22,2021-06-12,1,1.345473,0
